# Juvao AI — Qwen2.5-0.5B-Instruct Fine-tuning\n\nThis notebook fine-tunes a pretrained Qwen model with LoRA. It does not train a language model from scratch. Enable a Kaggle GPU in Notebook Settings for practical training.

In [ ]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes safetensors pyyaml

In [ ]:
import os, torch\nfrom pathlib import Path\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available(): print(torch.cuda.get_device_name(0))\nBASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'\nMAX_LENGTH = 1024\nOUTPUT_DIR = '/kaggle/working/juvao-lora'\nDATA_FILE = '/kaggle/input/YOUR_DATASET/conversations.jsonl'  # change this path\nUSE_4BIT = torch.cuda.is_available()  # set False if bitsandbytes is unsupported

## Dataset\nUpload a JSONL dataset as a Kaggle Dataset. Each line should contain `messages` with role/content entries, or `prompt` and `response`. The example below is only a smoke-test sample; replace it with a suitably sized, licensed dataset before training.

In [ ]:
import json\nif not Path(DATA_FILE).exists():\n    sample = [{'messages':[{'role':'user','content':'Hello!'}, {'role':'assistant','content':'Hi! How can I help you today?'}]}]\n    Path('/kaggle/working/sample.jsonl').write_text('\\n'.join(json.dumps(x) for x in sample))\n    DATA_FILE = '/kaggle/working/sample.jsonl'\nrows=[]; seen=set()\nfor line_no, line in enumerate(Path(DATA_FILE).read_text(encoding='utf-8').splitlines(), 1):\n    if not line.strip(): continue\n    obj=json.loads(line)\n    msgs=obj.get('messages') or [{'role':'user','content':obj['prompt']},{'role':'assistant','content':obj['response']}]\n    msgs=[{'role':m['role'].lower().strip(),'content':str(m['content']).strip()} for m in msgs if m.get('role','').lower().strip() in ('system','user','assistant') and str(m.get('content','')).strip()]\n    if not msgs or not any(m['role']=='assistant' for m in msgs): continue\n    key=json.dumps(msgs,sort_keys=True,ensure_ascii=False)\n    if key not in seen: seen.add(key); rows.append(msgs)\nassert rows, 'No valid conversations found. Check DATA_FILE.'\nprint('Unique conversations:', len(rows))

In [ ]:
from datasets import Dataset\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\ntokenizer=AutoTokenizer.from_pretrained(BASE_MODEL)\nif tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token\ntexts=[tokenizer.apply_chat_template(x, tokenize=False, add_generation_prompt=False) for x in rows]\ndataset=Dataset.from_dict({'text':texts}).train_test_split(test_size=0.05, seed=42) if len(texts)>20 else Dataset.from_dict({'text':texts}).train_test_split(test_size=0.0, seed=42)\ndef tokenize(batch): return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)\ntrain_ds=dataset['train'].map(tokenize, batched=True, remove_columns=['text'])\neval_ds=dataset['test'].map(tokenize, batched=True, remove_columns=['text']) if len(dataset['test']) else None\nuse_4bit=bool(USE_4BIT and torch.cuda.is_available())\nif use_4bit:\n    qconfig=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)\n    model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=qconfig,device_map='auto')\n    model=prepare_model_for_kbit_training(model)\nelse:\n    model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,torch_dtype='auto' if torch.cuda.is_available() else None)\nmodel.config.use_cache=False\nlora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],bias='none',task_type='CAUSAL_LM')\nmodel=get_peft_model(model,lora); model.print_trainable_parameters()

In [ ]:
args=TrainingArguments(output_dir=OUTPUT_DIR,num_train_epochs=2,per_device_train_batch_size=2,gradient_accumulation_steps=8,learning_rate=2e-4,warmup_ratio=0.03,logging_steps=5,save_steps=50,save_total_limit=2,gradient_checkpointing=True,fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),report_to='none',evaluation_strategy='steps' if eval_ds is not None else 'no',eval_steps=50 if eval_ds is not None else None)\ntrainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=eval_ds,data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False))\ntrainer.train()\ntrainer.save_model(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)\nprint('Saved adapter:', OUTPUT_DIR)

In [ ]:
from peft import PeftModel\nbase=AutoModelForCausalLM.from_pretrained(BASE_MODEL,torch_dtype='auto' if torch.cuda.is_available() else None,device_map='auto' if torch.cuda.is_available() else None)\ntuned=PeftModel.from_pretrained(base,OUTPUT_DIR)\ndef juvao_reply(prompt):\n    msgs=[{'role':'user','content':prompt}]\n    encoded=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors='pt').to(tuned.device)\n    generated=tuned.generate(encoded,max_new_tokens=160,do_sample=True,temperature=0.7,top_p=0.9,pad_token_id=tokenizer.pad_token_id)\n    return tokenizer.decode(generated[0][encoded.shape[1]:],skip_special_tokens=True)\nprint(juvao_reply('Give me three practical tips to improve spoken English.'))

## Export\nThe saved directory contains the PEFT adapter and tokenizer. Download it from the Kaggle Output panel or save it as a Kaggle Model. The adapter requires the original Qwen base model at inference time.

In [ ]:
import shutil\nshutil.make_archive('/kaggle/working/juvao-lora-adapter','zip',OUTPUT_DIR)\nprint('Adapter archive ready: /kaggle/working/juvao-lora-adapter.zip')